# Evaluation on gemma-3-4b-it using BBEH benchmark for reasoning tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.1 MB/s eta 0:00:00


In [2]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "google/gemma-3-4b-it"

In [5]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model= AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [7]:
def load_json(path):
  with open(path,"r") as f:
    data = json.load(f)

    df = pd.DataFrame(data['examples'])
    return df

In [8]:
def get_prompt(instruction,questions):
  prompts=[]
  for i in range(len(questions)):
    question = questions[i].split("Reply")[0]
    prompt = f"{instruction}\n\n{question}\n\nAnswer: "
    prompts.append(prompt)
  return prompts

In [22]:
def training(prompts,num_tokens=1):
    predictions = []
    for i, prompt in enumerate(prompts):
        chat = [
            {"role": "user", "content": prompt}
        ]
        formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

        input_ids = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

        output_tokens = model.generate(
            **input_ids,
            max_new_tokens=num_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        new_tokens = output_tokens[0][len(input_ids[0]):]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        print(f"Iteration {i}")
        print(f"Model Output: {response}\n")

        predictions.append(response)

    return predictions

## Task Casual Understanding

In [10]:
dataset = load_json("/content/task_casual_understanding.json")

In [11]:
dataset

,input,target
0,Question: Lauren and Jane work for the same co...,Yes
1,Question: Lauren and Jane work for the same co...,Yes
2,Question: The receptionist in the philosophy d...,Yes
3,Question: The receptionist in the philosophy d...,No
4,Question: Claire's parents bought her an old c...,Yes
...,...,...
195,Question: A test tube contains a mixture. Sam ...,Yes
196,Question: Reagent X and Y are both added to a ...,No
197,Question: There is a test tube on the left and...,Yes
198,Question: Sam is heating Reagent X in a test t...,Yes


In [12]:
questions = dataset['input'].values.tolist()

In [13]:
true_answers = dataset['target'].values.tolist()

In [14]:
instruction = f"""Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:
Yes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.
"""

In [15]:
prompts = get_prompt(instruction,questions)

In [16]:
prompts[0]

"Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:\nYes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.\n\n\nQuestion: Lauren and Jane work for the same company. They each need to use a computer for work sometimes. Unfortunately, the computer isn't very powerful. If two people are logged on at the same time, it usually crashes. So the company decided to institute an official policy. It declared that Lauren would be the only one permitted to use the computer in the mornings and that Jane would be the only one permitted to use the computer in the afternoons. As expected, Lauren logged on the computer the next day at 9:00 am. But Jane decided to disobey the official policy. She also logged on at 9:00 am. The computer crashed immediately. Did Jane cause the computer to crash

In [23]:
predictions = training(prompts,num_tokens=5)

Iteration 0
Model Output: Yes

Iteration 1
Model Output: Yes

Iteration 2
Model Output: Yes

Iteration 3
Model Output: Yes

Iteration 4
Model Output: Yes

Iteration 5
Model Output: Yes

Iteration 6
Model Output: Ambiguous

Iteration 7
Model Output: Ambiguous

Iteration 8
Model Output: Yes

Iteration 9
Model Output: Yes

Iteration 10
Model Output: Yes

Iteration 11
Model Output: Yes

Iteration 12
Model Output: No

Iteration 13
Model Output: Ambiguous

Iteration 14
Model Output: Yes

Iteration 15
Model Output: Yes

Iteration 16
Model Output: Yes

Iteration 17
Model Output: Yes

Iteration 18
Model Output: Yes

Iteration 19
Model Output: Yes

Iteration 20
Model Output: Yes

Iteration 21
Model Output: Yes

Iteration 22
Model Output: Yes

Iteration 23
Model Output: Yes

Iteration 24
Model Output: Yes

Iteration 25
Model Output: Yes

Iteration 26
Model Output: No

Iteration 27
Model Output: Yes

Iteration 28
Model Output: Yes

Iteration 29
Model Output: Yes

Iteration 30
Model Output: Yes

It

In [24]:
predictions[:10]

['Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'Ambiguous',
 'Ambiguous',
 'Yes',
 'Yes']

In [25]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

   Ambiguous       0.27      0.07      0.11        45
          No       0.71      0.06      0.11        80
         Yes       0.39      0.95      0.55        75

    accuracy                           0.40       200
   macro avg       0.46      0.36      0.26       200
weighted avg       0.49      0.40      0.28       200



## Task Movie Recommendation

In [26]:
dataset = load_json("/content/task_movie_recommendation.json")

In [27]:
dataset

,input,target
0,Which option has more similar movies in terms ...,(E)
1,Which option has more similar movies in terms ...,(C)
2,Which option has more similar movies in terms ...,(C)
3,Which option has more similar movies in terms ...,(A)
4,Which option has more similar movies in terms ...,(I)
...,...,...
195,Which option has more similar movies in terms ...,(E)
196,Which option has more similar movies in terms ...,(E)
197,Which option has more similar movies in terms ...,(E)
198,Which option has more similar movies in terms ...,(A)


In [28]:
dataset["target"].value_counts()

,count
target,
(A),26
(G),22
(H),21
(E),21
(J),20
(D),20
(C),18
(F),18
(B),17


In [29]:
questions = dataset['input'].values.tolist()

In [30]:
true_answers = dataset['target'].values.tolist()

In [31]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'C', 'A', 'I', 'D', 'B', 'F', 'D', 'H']

In [44]:
instruction = f"""You are an assistant that determines which group of movies is most similar in terms of what people are likely to enjoy together. Reply only with the letter (A, B, C, D, E, F, G, H, I, or J) of the group that is most similar.
Do not write any explanation, full movie names, or extra text. The answer must be a single uppercase letter."""

In [45]:
prompts = get_prompt(instruction,questions)

In [46]:
prompts[1]

'You are an assistant that determines which group of movies is most similar in terms of what people are likely to enjoy together. Reply only with the letter (A, B, C, D, E, F, G, H, I, or J) of the group that is most similar.\nDo not write any explanation, full movie names, or extra text. The answer must be a single uppercase letter.\n\nWhich option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) Street Fighter II The Animated Movie, The Sixth Sense, The Matrix, The Shawshank Redemption, Terminator 2 Judgment Day\n(B) The Boy Who Could Fly, The Sheltering Sky, Forrest Gump, Terminator 2 Judgment Day, The Matrix\n(C) Forrest Gump, The Shawshank Redemption, Terminator 2 Judgment Day, The Matrix, The Sixth Sense\n(D) The Boy Who Could Fly, The Sheltering Sky, Street Fighter II The Animated Movie, The Sixth Sense, Terminator 2 Judgment Day\n(E) The Matrix, The Boy Who Could Fly, Forrest Gump, The Shawshank Redemption, The Sixth Sense\n(F)

In [47]:
predictions = training(prompts,num_tokens=5)

Iteration 0
Model Output: A

Iteration 1
Model Output: C

Iteration 2
Model Output: C

Iteration 3
Model Output: A

Iteration 4
Model Output: B

Iteration 5
Model Output: B

Iteration 6
Model Output: B

Iteration 7
Model Output: B

Iteration 8
Model Output: C

Iteration 9
Model Output: J

Iteration 10
Model Output: A

Iteration 11
Model Output: A

Iteration 12
Model Output: A

Iteration 13
Model Output: A

Iteration 14
Model Output: B

Iteration 15
Model Output: J

Iteration 16
Model Output: B

Iteration 17
Model Output: C

Iteration 18
Model Output: A

Iteration 19
Model Output: B

Iteration 20
Model Output: A

Iteration 21
Model Output: A

Iteration 22
Model Output: C

Iteration 23
Model Output: H

Iteration 24
Model Output: A

Iteration 25
Model Output: A

Iteration 26
Model Output: B

Iteration 27
Model Output: A

Iteration 28
Model Output: A

Iteration 29
Model Output: A

Iteration 30
Model Output: C

Iteration 31
Model Output: C

Iteration 32
Model Output: A

Iteration 33
Model O

In [48]:
predictions[:10]

['A', 'C', 'C', 'A', 'B', 'B', 'B', 'B', 'C', 'J']

In [49]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.20      0.73      0.32        26
           B       0.16      0.29      0.20        17
           C       0.14      0.33      0.20        18
           D       0.67      0.10      0.17        20
           E       0.00      0.00      0.00        21
           F       0.25      0.06      0.09        18
           G       1.00      0.14      0.24        22
           H       0.38      0.14      0.21        21
           I       0.33      0.06      0.10        17
           J       0.27      0.15      0.19        20

    accuracy                           0.21       200
   macro avg       0.34      0.20      0.17       200
weighted avg       0.35      0.21      0.18       200



## Task Disambiguation QA

In [52]:
dataset = load_json("/content/task_disambiguationQA.json")

In [53]:
dataset['target'].value_counts()

,count
target,
(B),25
(D),24
(C),23
(A),22
(E),17
(F),7
(G),1
(H),1


In [67]:
dataset

,input,target
0,Here is a sentence with pronoun(s) whose antec...,(A)
1,Here is a sentence with pronoun(s) whose antec...,(D)
2,Here is a sentence with pronoun(s) whose antec...,(E)
3,Here is a sentence with pronoun(s) whose antec...,(B)
4,Here is a sentence with pronoun(s) whose antec...,(A)
...,...,...
115,Here is a sentence with pronoun(s) whose antec...,(D)
116,Here is a sentence with pronoun(s) whose antec...,(A)
117,Here is a sentence with pronoun(s) whose antec...,(B)
118,Here is a sentence with pronoun(s) whose antec...,(D)


In [54]:
questions = dataset['input'].values.tolist()

In [68]:
questions[0]

'Here is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The visitor tells the teacher that she likes the cake, and then tells the chef that it was her favorite dish.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) The visitor likes cake and the visitor told the chef.\n(B) The teacher likes cake and the teacher told the chef.\n(C) The visitor likes cake and the teacher told the chef.\n(D) The teacher likes cake and the visitor told the chef.\n(E) Ambiguous.'

In [55]:
true_answers = dataset['target'].values.tolist()

In [56]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['A', 'D', 'E', 'B', 'A', 'B', 'B', 'A', 'C', 'D']

In [69]:
instruction = f"""Instruction: You are a linguistics expert specializing in coreference resolution.
Task: Read the sentence and the provided options. Identify the correct antecedent(s) for the pronouns based strictly on the context and causal logic provided.
Constraint: Output ONLY the uppercase letter corresponding to the correct option. Do not explain your reasoning.
"""

In [70]:
prompts = get_prompt(instruction,questions)

In [71]:
prompts[1]

'Instruction: You are a linguistics expert specializing in coreference resolution. \nTask: Read the sentence and the provided options. Identify the correct antecedent(s) for the pronouns based strictly on the context and causal logic provided.\nConstraint: Output ONLY the uppercase letter corresponding to the correct option. Do not explain your reasoning.\n\n\nHere is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The designer called the janitor and asked him to clean the room as soon as possible, but he went to fetch a cup of coffee first and then returned to his desk.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) Designer is cleaning the room as soon as possible and got coffee.\n(B) Janitor is cleaning the room as soon as possible and got coffee.\n(C) Designer is cleaning the room as soon as possible and janitor got coffee.\n(D) Janitor is cleaning the room as soon as pos

In [74]:
predictions = training(prompts,num_tokens=5)

Iteration 0
Model Output: B

Iteration 1
Model Output: B

Iteration 2
Model Output: B

Iteration 3
Model Output: B

Iteration 4
Model Output: B

Iteration 5
Model Output: B

Iteration 6
Model Output: B

Iteration 7
Model Output: C

Iteration 8
Model Output: B

Iteration 9
Model Output: B

Iteration 10
Model Output: C

Iteration 11
Model Output: C

Iteration 12
Model Output: B

Iteration 13
Model Output: B

Iteration 14
Model Output: B

Iteration 15
Model Output: C

Iteration 16
Model Output: D

Iteration 17
Model Output: C

Iteration 18
Model Output: C

Iteration 19
Model Output: B

Iteration 20
Model Output: B

Iteration 21
Model Output: D

Iteration 22
Model Output: B

Iteration 23
Model Output: C

Iteration 24
Model Output: B

Iteration 25
Model Output: B

Iteration 26
Model Output: B

Iteration 27
Model Output: B

Iteration 28
Model Output: A

Iteration 29
Model Output: B

Iteration 30
Model Output: C

Iteration 31
Model Output: E

Iteration 32
Model Output: B

Iteration 33
Model O

In [75]:
predictions

['B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'C',
 'B',
 'B',
 'C',
 'C',
 'B',
 'B',
 'B',
 'C',
 'D',
 'C',
 'C',
 'B',
 'B',
 'D',
 'B',
 'C',
 'B',
 'B',
 'B',
 'B',
 'A',
 'B',
 'C',
 'E',
 'B',
 'B',
 'B',
 'C',
 'A',
 'B',
 'A',
 'C',
 'B',
 'B',
 'B',
 'B',
 'C',
 'C',
 'D',
 'C',
 'B',
 'B',
 'C',
 'A',
 'B',
 'B',
 'B',
 'C',
 'D',
 'A',
 'B',
 'E',
 'C',
 'B',
 'A',
 'B',
 'B',
 'C',
 'B',
 'C',
 'C',
 'B',
 'C',
 'B',
 'B',
 'B',
 'B',
 'A',
 'C',
 'B',
 'A',
 'B',
 'B',
 'C',
 'B',
 'B',
 'C',
 'B',
 'B',
 'A',
 'B',
 'B',
 'A',
 'E',
 'B',
 'B',
 'C',
 'C',
 'B',
 'C',
 'B',
 'B',
 'B',
 'C',
 'B',
 'B',
 'B',
 'B',
 'C',
 'B',
 'C',
 'B',
 'B',
 'B',
 'C',
 'D',
 'C',
 'B',
 'C',
 'B',
 'E',
 'B']

In [76]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.20      0.09      0.12        22
           B       0.23      0.64      0.34        25
           C       0.25      0.35      0.29        23
           D       0.60      0.12      0.21        24
           E       0.25      0.06      0.10        17
           F       0.00      0.00      0.00         7
           G       0.00      0.00      0.00         1
           H       0.00      0.00      0.00         1

    accuracy                           0.25       120
   macro avg       0.19      0.16      0.13       120
weighted avg       0.29      0.25      0.20       120



## Task: Boolean Expressions

In [77]:
dataset = load_json("/content/task_boolean_expressions.json")

In [78]:
dataset

,input,target
0,"From the following five expressions, only one ...",(E)
1,"From the following five expressions, only one ...",(C)
2,"From the following five expressions, only one ...",(E)
3,"From the following five expressions, only one ...",(D)
4,"From the following five expressions, only one ...",(C)
...,...,...
195,"From the following five expressions, only one ...",(A)
196,"From the following five expressions, only one ...",(B)
197,"From the following five expressions, only one ...",(B)
198,"From the following five expressions, only one ...",(C)


In [79]:
dataset["target"].value_counts()

,count
target,
(E),43
(B),41
(C),40
(A),39
(D),37


In [80]:
questions = dataset["input"].values.tolist()

In [81]:
true_answers = dataset["target"].values.tolist()

In [83]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'E', 'D', 'C', 'D', 'C', 'D', 'A', 'A']

In [82]:
questions[0]

'From the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not (not (not (False) or (-5 - (1 / -5) is greater than -7)) or (not (False) or (-5 * -10 + -5 * -6 is greater than -4 * 6))) or (not (not (False) or (False)) or (not not not (-3 - (-2 / 4) is greater than -6) and not not (The capital of India is Mumbai.)))) and (not (not (not (The capital of the Nigeria is Abuja.) or (-7 - (-4 / 7) > 4)) or (not (-6 - (-7 / -5) > -6) and (True))) and (not not ((-1 * 9 is less than or equal to -6) or (-4 * 10 + -1 * -7 <= 9 * 10)) or not not not (not not (6 * -2 is less than or equal to 4) or not not not (-5 * 9 + 6 * -3 <= -9 * 6))))\n(B) ((not (not not not (not (not (not not not (not not (not (not (max(-3, 8, 5, -2) - min(-3, 8, 5, -2) <= 4) and (False)) or (max(-6, 8, 5, -1) - min(-6, 8, 5, -1) <= 9)) or not not not (-6 - (9 / -1) is less than or equal to 6)) and not not (The capital of Afghanistan is Kabul.)) or (1 - (5 / -2) > 4)) or (True)) and not not 

In [ ]:
instruction = f"""Instruction: You are a high-level logic and reasoning engine.
Task: Carefully evaluate the five nested Boolean expressions (A, B, C, D, E) provided.
Method:
1. Break down each expression into its sub-components.
2. Evaluate the truth value of the geographic facts (Capitals).
3. Solve the mathematical inequalities.
4. Simplify the 'not' operators step-by-step.
Constraint: After your internal evaluation, identify the ONLY expression that is True.
Output Format: Output exactly one uppercase letter (A, B, C, D, or E) as the final result."""

In [84]:
prompts = get_prompt(instruction,questions)

In [85]:
prompts[0]

'Instruction: You are a linguistics expert specializing in coreference resolution. \nTask: Read the sentence and the provided options. Identify the correct antecedent(s) for the pronouns based strictly on the context and causal logic provided.\nConstraint: Output ONLY the uppercase letter corresponding to the correct option. Do not explain your reasoning.\n\n\nFrom the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not (not (not (False) or (-5 - (1 / -5) is greater than -7)) or (not (False) or (-5 * -10 + -5 * -6 is greater than -4 * 6))) or (not (not (False) or (False)) or (not not not (-3 - (-2 / 4) is greater than -6) and not not (The capital of India is Mumbai.)))) and (not (not (not (The capital of the Nigeria is Abuja.) or (-7 - (-4 / 7) > 4)) or (not (-6 - (-7 / -5) > -6) and (True))) and (not not ((-1 * 9 is less than or equal to -6) or (-4 * 10 + -1 * -7 <= 9 * 10)) or not not not (not not (6 * -2 is less than or equal to 4) or not not not

In [86]:
predictions = training(prompts,num_tokens=5)

Iteration 0
Model Output: E

Iteration 1
Model Output: C

Iteration 2
Model Output: C

Iteration 3
Model Output: C

Iteration 4
Model Output: C

Iteration 5
Model Output: D

Iteration 6
Model Output: C

Iteration 7
Model Output: C

Iteration 8
Model Output: A

Iteration 9
Model Output: C

Iteration 10
Model Output: C

Iteration 11
Model Output: C

Iteration 12
Model Output: C

Iteration 13
Model Output: C

Iteration 14
Model Output: C

Iteration 15
Model Output: C

Iteration 16
Model Output: A

Iteration 17
Model Output: A

Iteration 18
Model Output: C

Iteration 19
Model Output: C

Iteration 20
Model Output: C

Iteration 21
Model Output: C

Iteration 22
Model Output: C

Iteration 23
Model Output: C

Iteration 24
Model Output: E

Iteration 25
Model Output: A

Iteration 26
Model Output: D

Iteration 27
Model Output: C

Iteration 28
Model Output: C

Iteration 29
Model Output: D

Iteration 30
Model Output: C

Iteration 31
Model Output: C

Iteration 32
Model Output: C

Iteration 33
Model O

In [87]:
predictions

['E',
 'C',
 'C',
 'C',
 'C',
 'D',
 'C',
 'C',
 'A',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'A',
 'A',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'E',
 'A',
 'D',
 'C',
 'C',
 'D',
 'C',
 'C',
 'C',
 'C',
 'A',
 'C',
 'E',
 'C',
 'C',
 'C',
 'D',
 'C',
 'A',
 'E',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C',
 'A',
 'A',
 'C',
 'C',
 'A',
 'C',
 'C',
 'A',
 'C',
 'C',
 'C',
 'C',
 'D',
 'E',
 'A',
 'C',
 'E',
 'D',
 'C',
 'D',
 'C',
 'D',
 'B',
 'C',
 'D',
 'C',
 'E',
 'B',
 'C',
 'C',
 'A',
 'A',
 'B',
 'C',
 'C',
 'C',
 'E',
 'A',
 'C',
 'A',
 'C',
 'A',
 'C',
 'C',
 'A',
 'C',
 'C',
 'C',
 'C',
 'A',
 'D',
 'C',
 'A',
 'C',
 'E',
 'D',
 'B',
 'D',
 'C',
 'E',
 'C',
 'C',
 'D',
 'C',
 'C',
 'C',
 'C',
 'D',
 'C',
 'C',
 'D',
 'C',
 'C',
 'D',
 'C',
 'A',
 'D',
 'C',
 'A',
 'C',
 'D',
 'A',
 'C',
 'C',
 'C',
 'A',
 'C',
 'A',
 'C',
 'C',
 'C',
 'A',
 'C',
 'B',
 'A',
 'C',
 'C',
 'A',
 'A',
 'C',
 'C',
 'C',
 'A',
 'D',
 'D',
 'C',
 'C',
 'C',
 'C',
 'C',
 'C'

In [88]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.11      0.10      0.11        39
           B       0.33      0.05      0.09        41
           C       0.23      0.68      0.34        40
           D       0.21      0.14      0.16        37
           E       0.40      0.14      0.21        43

    accuracy                           0.22       200
   macro avg       0.26      0.22      0.18       200
weighted avg       0.26      0.22      0.18       200

